## prepare and inspect before connecting to LLM

This notebook answers the question "Does retrieval quality affect downstream LLM-based ATT&CK mapping accuracy?"

For each TRAM dataset sentence:
- goes through the 3 retrieval method (BoW, TF-IDF, embedding)
- We get our top 5 candidates
- go through same LLM, same prompt
- select ONE candidate only
- compare with the ground truth

In [12]:
import pandas as pd

tram_df = pd.read_csv(
    "/Users/vivianwu/Desktop/NLP_Research/data/tram_single_label_compatible.csv"
)

test_df = tram_df[
    tram_df["split"] == "test"
].copy()

print("Test examples:", len(test_df))
print(
    "Test techniques:",
    test_df["technique_id"].nunique()
)

Test examples: 511
Test techniques: 47


In [13]:
#load mitre and embedding model
import numpy as np
import re

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


mitre_df = pd.read_csv(
    "/Users/vivianwu/Desktop/NLP_Research/data/mitre_techniques.csv"
)

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

W1117 07:54:41.074000 62429 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [14]:
#same cleaning function from our embedding experiment
def clean_mitre_text(text):
    text = str(text)

    text = re.sub(
        r"\(Citation:[^)]+\)",
        "",
        text
    )

    text = re.sub(
        r"\[([^\]]+)\]\([^)]+\)",
        r"\1",
        text
    )

    text = text.replace("`", "")
    text = re.sub(r"\s+", " ", text)

    return text.strip()


mitre_df["retrieval_text"] = (
    mitre_df["name"].fillna("")
    + ". "
    + mitre_df["description"].fillna("")
)

mitre_df["retrieval_text"] = (
    mitre_df["retrieval_text"]
    .apply(clean_mitre_text)
)

In [15]:
#MITRE embedding
mitre_embeddings = model.encode(
    mitre_df["retrieval_text"].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

In [16]:
#TRAM test queries
test_df["query_text"] = (
    test_df["sentence"]
    .fillna("")
    .str.strip()
)

query_embeddings = model.encode(
    test_df["query_text"].tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

In [17]:
#similarity
similarity_matrix = cosine_similarity(
    query_embeddings,
    mitre_embeddings
)

In [18]:
#get our top 5 candidates
top5_candidates = []

for i in range(len(test_df)):

    scores = similarity_matrix[i]

    top_indices = np.argsort(
        scores
    )[::-1][:5]

    candidates = []

    for idx in top_indices:

        candidates.append({
            "technique_id":
                mitre_df.iloc[idx]["technique_id"],

            "name":
                mitre_df.iloc[idx]["name"],

            "description":
                mitre_df.iloc[idx]["retrieval_text"],

            "score":
                scores[idx]
        })

    top5_candidates.append(candidates)

In [19]:
test_df["embedding_candidates"] = (
    top5_candidates
)

In [20]:
#inspect
example = test_df.iloc[0]

print("THREAT:")
print(example["sentence"])

print("\nGROUND TRUTH:")
print(example["technique_id"])

print("\nCANDIDATES:")

for candidate in example[
    "embedding_candidates"
]:
    print(
        candidate["technique_id"],
        "-",
        candidate["name"]
    )

THREAT:
Similar functionality was linked in the past to LockBit 2.0, although the Rorschach Ransomware GPO deployment is carried out differently, as described below: Rorschach copies its files into the scripts folder of the DC, and deletes them from the original location.

GROUND TRUTH:
T1070.004

CANDIDATES:
T1003.002 - Security Account Manager
T1552.006 - Group Policy Preferences
T1490 - Inhibit System Recovery
T1484.001 - Group Policy Modification
T1003.006 - DCSync


**- now assess if the correct ATT&CK technique somewhere in those Top 5 candidates**

In [21]:
def contains_ground_truth(candidates, ground_truth):
    """this function checks the candidates"""
    return any(
        candidate["technique_id"] == ground_truth
        for candidate in candidates
    )

In [22]:
#apply it to test queries
test_df[
    "embedding_retrieval_success"
] = test_df.apply(
    lambda row: contains_ground_truth(
        row["embedding_candidates"],
        row["technique_id"]
    ),
    axis=1
)

In [23]:
test_df[
    [
        "sentence",
        "technique_id",
        "embedding_retrieval_success"
    ]
].head(10)

,sentence,technique_id,embedding_retrieval_success
2566,Similar functionality was linked in the past t...,T1070.004,False
2567,Rorschach then creates a group policy (see App...,T1484.001,False
2568,The ransomware creates another group policy in...,T1484.001,False
2569,This is done by creating a schedule task invok...,T1053.005,True
2570,"First, the initial loader/injector winutils.dl...",T1027,False
2571,"After Rorschach is injected into notepad.exe, ...",T1055,False
2572,It makes direct system calls using the “syscal...,T1106,True
2573,The ransomware finds the relevant syscall numb...,T1106,False
2574,"When needed, it calls a stub routine that uses...",T1106,True
2575,"In other words, the malware first creates a sy...",T1106,False


In [24]:
#count our success and failures
test_df[
    "embedding_retrieval_success"
].value_counts()

embedding_retrieval_success
False    314
True     197
Name: count, dtype: int64

In [14]:
#check
print(
    test_df[
        "embedding_retrieval_success"
    ].mean()
)

0.38551859099804303


- On the held-out test set, embeddings retrieved the correct ATT&CK technique among the Top 5 candidates for 38.5% of queries.

- now we have:

1. Threat description
2. Embedding retrieval
3. Top 5 candidates
4. Correct answer present?
5. True / False

**inspect 1 prompt before we move on to API**

In [10]:
def build_prompt(threat_sentence, candidates):

    candidate_text = ""

    for i, candidate in enumerate(candidates, start=1):

        candidate_text += (
            f"\nCandidate {i}\n"
            f"ATT&CK ID: {candidate['technique_id']}\n"
            f"Name: {candidate['name']}\n"
            f"Description: {candidate['description']}\n"
        )

    prompt = f"""
You are performing MITRE ATT&CK technique mapping.

Cyber threat description:
{threat_sentence}

Candidate MITRE ATT&CK techniques:
{candidate_text}

Select the ONE candidate that best represents the
adversary behavior described in the cyber threat description.

You must select from the provided candidates.

Return only the ATT&CK technique ID.
"""

    return prompt.strip()

In [19]:
#test without calling ll
example = test_df.iloc[0]

example_prompt = build_prompt(
    example["sentence"],
    example["embedding_candidates"]
)

print(example_prompt)

You are performing MITRE ATT&CK technique mapping.

Cyber threat description:
Similar functionality was linked in the past to LockBit 2.0, although the Rorschach Ransomware GPO deployment is carried out differently, as described below: Rorschach copies its files into the scripts folder of the DC, and deletes them from the original location.

Candidate MITRE ATT&CK techniques:

Candidate 1
ATT&CK ID: T1003.002
Name: Security Account Manager
Description: Security Account Manager. Adversaries may attempt to extract credential material from the Security Account Manager (SAM) database either through in-memory techniques or through the Windows Registry where the SAM database is stored. The SAM is a database file that contains local accounts for the host, typically those found with the <code>net user</code> command. Enumerating the SAM database requires SYSTEM level access. A number of tools can be used to retrieve the SAM file through in-memory techniques: * pwdumpx.exe * gsecdump * Mimikatz

**verified the result contains what LLM needs:**

- The original TRAM threat description
- 5 candidates retrieved 
- odel select one of the candidates and return only its ID

**before API calls, create a small test sample**

In [25]:
llm_sample = test_df.head(10).copy()

In [26]:
success_sample = test_df[
    test_df["embedding_retrieval_success"]
].head(5)

failure_sample = test_df[
    ~test_df["embedding_retrieval_success"]
].head(5)

llm_sample = pd.concat(
    [
        success_sample,
        failure_sample
    ],
    ignore_index=True
)

now we will have:
- 5 where correct candidate is available
- 5 where correct candidate is not available

In [27]:
print(
    test_df[
        "embedding_retrieval_success"
    ].value_counts()
)

embedding_retrieval_success
False    314
True     197
Name: count, dtype: int64


In [28]:
print(
    f"Embedding test Recall@5: "
    f"{test_df['embedding_retrieval_success'].mean():.3f}"
)

Embedding test Recall@5: 0.386


**Next steps:**

- connect an LLM and make predictions
- Tried Open AI, had trouble connecting, using Google

In [2]:
#create client
from google import genai

client = genai.Client()

In [3]:
from google import genai

client = genai.Client()

for model in client.models.list():
    if "generateContent" in model.supported_actions:
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3.1-flash-lite
models/gemini-3-pro-image
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/gemini-4-carbon
models/gemini-4-hydrogen
models/antigravity-preview-05-2026
models/antigravity-preview-09-2026
models/antigravity-preview-latest
models/deep-research-max-preview-04-2026
models/deep-research-preview-04-2026
models/deep-research-pro-preview-12-2025


In [4]:
response = client.models.generate_content(
    model="models/gemini-2.5-flash",
    contents="Say hello."
)

print(response.text)

Hello!


- tested a few models, "models/gemini-2.5-flash" is the one that works for me, while some other gave a 503 error

In [5]:
#same model for all experiement
LLM_MODEL = "models/gemini-2.5-flash"

In [6]:
def call_llm(prompt):
    response = client.models.generate_content(
        model=LLM_MODEL,
        contents=prompt
    )

    return response.text.strip()

In [44]:
#test 1 example
example = llm_sample.iloc[0]

prompt = build_prompt(
    example["sentence"],
    example["embedding_candidates"]
)

raw_response = call_llm(prompt)

print("Raw response:", raw_response)
print("Ground truth:", example["technique_id"])
print(
    "Retrieval success:",
    example["embedding_retrieval_success"]
)

Raw response: T1053.005
Ground truth: T1053.005
Retrieval success: True


In [29]:
#parse attack id
import re

def extract_attack_id(text):
    match = re.search(
        r"\bT\d{4}(?:\.\d{3})?\b",
        text
    )

    if match:
        return match.group(0)

    return None

In [30]:
prediction = extract_attack_id(
    raw_response
)

correct = (
    prediction == example["technique_id"]
)

print("Prediction:", prediction)
print("Ground truth:", example["technique_id"])
print("Correct:", correct)

Prediction: T1216.001
Ground truth: T1070.004
Correct: False


In [47]:
#double checking if Gemini use the candidates
candidate_ids = [
    candidate["technique_id"]
    for candidate in example["embedding_candidates"]
]

print("Candidates:", candidate_ids)
print(
    "Prediction is a candidate:",
    prediction in candidate_ids
)

Candidates: ['T1053.005', 'T1055.012', 'T1529', 'T1053', 'T1055.003']
Prediction is a candidate: True


- going to send llm sample to gemini
- there will be:
5 examples where embedding retrieval succeeded — correct technique is in Top 5.
5 examples where embedding retrieval failed — correct technique is not in Top 5.

we will be able to verify if:
-  Gemini select the correct candidate?
- Confirm what happens when the correct answer isn't available to Gemini.

In [53]:
#send to gemini
#Threat sentence -> Top-5 embedding candidates -> build_prompt() -> Gemini 2.5 Flash -> Raw answer
# extract_attack_id() -> Predicted ATT&CK ID
import time

predictions = []
raw_responses = []

for i, row in llm_sample.iterrows():

    # Build the prompt using the threat sentence
    # and its Top-5 embedding candidates
    prompt = build_prompt(
        row["sentence"],
        row["embedding_candidates"]
    )

    # Send the prompt to Gemini
    raw_response = call_llm(prompt)

    # Extract the ATT&CK ID from Gemini's response
    prediction = extract_attack_id(
        raw_response
    )

    # Save both versions
    raw_responses.append(raw_response)
    predictions.append(prediction)

    print(
        f"Completed {len(predictions)}/10"
    )

    # Stay below the API's 5 requests/minute limit
    time.sleep(15)

Completed 1/10
Completed 2/10
Completed 3/10
Completed 4/10
Completed 5/10
Completed 6/10
Completed 7/10
Completed 8/10
Completed 9/10
Completed 10/10


- had a lot of trouble with Gemini's daily and running quota. this part is assisted by ChatGPT to connect with LLM

In [60]:
llm_sample["llm_raw_response"] = raw_responses
llm_sample["llm_prediction"] = predictions

In [61]:
llm_sample[
    [
        "technique_id",
        "llm_prediction"
    ]
]

,technique_id,llm_prediction
0,T1053.005,T1053.005
1,T1106,T1106
2,T1106,T1106
3,T1059.003,T1059.003
4,T1027,T1027.008
5,T1070.004,T1484.001
6,T1484.001,T1037.003
7,T1484.001,T1679
8,T1027,T1055.001
9,T1055,T1546.012


In [62]:
#calculate correctedness
llm_sample["llm_correct"] = (
    llm_sample["llm_prediction"]
    == llm_sample["technique_id"]
)

In [63]:
llm_sample[
    [
        "technique_id",
        "embedding_retrieval_success",
        "llm_prediction",
        "llm_correct"
    ]
]

,technique_id,embedding_retrieval_success,llm_prediction,llm_correct
0,T1053.005,True,T1053.005,True
1,T1106,True,T1106,True
2,T1106,True,T1106,True
3,T1059.003,True,T1059.003,True
4,T1027,True,T1027.008,False
5,T1070.004,False,T1484.001,False
6,T1484.001,False,T1037.003,False
7,T1484.001,False,T1679,False
8,T1027,False,T1055.001,False
9,T1055,False,T1546.012,False


In [64]:
#save results first, gemini quota keeps running out. so i wont have to rerun them aagin
llm_sample.to_pickle(
    "llm_embedding_pilot.pkl"
)

llm_sample.to_csv(
    "llm_embedding_pilot.csv",
    index=False
)

print("Pilot results saved.")

Pilot results saved.


**Overall pilot accuracy**
I got:
`4/10= 40%`

- When the correct technique was available among the retrieved candidates, Gemini selected it correctly in 4 of 5 pilot cases. When retrieval failed, none of the predictions were correct.

In [32]:
#create a fix LLM evaluation sample
evaluation_sample = test_df.sample(
    n=20,
    random_state=42
).copy()

print("Evaluation examples:", len(evaluation_sample))

Evaluation examples: 20


In [33]:
print(
    evaluation_sample[
        "embedding_retrieval_success"
    ].value_counts()
)

embedding_retrieval_success
False    10
True     10
Name: count, dtype: int64


- save the evaluation sample before making API calls, so i have the exact same 20 threat descriptions for BoW, TF-IDF, and embeddings.

In [34]:
evaluation_sample.to_pickle(
    "llm_evaluation_sample.pkl"
)

evaluation_sample.to_csv(
    "llm_evaluation_sample.csv",
    index=False
)

print("Evaluation sample saved.")

Evaluation sample saved.


In [70]:
embedding_eval = evaluation_sample.copy()

In [71]:
#store gemini result
embedding_eval["llm_raw_response"] = None
embedding_eval["llm_prediction"] = None
embedding_eval["llm_correct"] = False
embedding_eval["valid_candidate"] = False

In [72]:
embedding_eval[
    [
        "technique_id",
        "embedding_retrieval_success",
        "llm_prediction",
        "llm_correct"
    ]
].head()

,technique_id,embedding_retrieval_success,llm_prediction,llm_correct
2690,T1105,False,None,False
2650,T1003.001,False,None,False
2999,T1566.001,True,None,False
2821,T1059.003,False,None,False
2634,T1105,True,None,False


In [45]:
def prediction_in_candidates(prediction, candidates):
    candidate_ids = [
        candidate["technique_id"]
        for candidate in candidates
    ]

    return prediction in candidate_ids

In [76]:
#now run gemini 1 query at a time and then save
import time

for i, row in embedding_eval.iterrows():

    # Skip rows already completed
    if pd.notna(row["llm_prediction"]):
        continue

    # Build the prompt
    prompt = build_prompt(
        row["sentence"],
        row["embedding_candidates"]
    )

    try:
        # Call Gemini
        raw_response = call_llm(prompt)

        # Extract predicted ATT&CK ID
        prediction = extract_attack_id(
            raw_response
        )

        # Check whether prediction came from Top-5 candidates
        valid = prediction_in_candidates(
            prediction,
            row["embedding_candidates"]
        )

        # Compare with ground truth
        correct = (
            prediction == row["technique_id"]
        )

        # Store results
        embedding_eval.at[i, "llm_raw_response"] = raw_response
        embedding_eval.at[i, "llm_prediction"] = prediction
        embedding_eval.at[i, "valid_candidate"] = valid
        embedding_eval.at[i, "llm_correct"] = correct

        # save
        embedding_eval.to_pickle(
            "embedding_llm_results.pkl"
        )

        print(
            f"Completed {i}: "
            f"prediction={prediction}, "
            f"correct={correct}"
        )

        # Stay below Gemini's per-minute limit
        time.sleep(15)

    except Exception as e:

        print(f"Stopped at row {i}")
        print("Error:", e)

        # Save everything completed so far
        embedding_eval.to_pickle(
            "embedding_llm_results.pkl"
        )

        break

Completed 2965: prediction=T1595.002, correct=False
Completed 2644: prediction=T1673, correct=False
Completed 2908: prediction=T1218.001, correct=False
Completed 2621: prediction=T1059.001, correct=False
Completed 2639: prediction=T1190, correct=True
Completed 2742: prediction=T1020, correct=False


In [77]:
completed = embedding_eval[
    embedding_eval["llm_prediction"].notna()
]

print("Completed:", len(completed))
print("Remaining:", len(embedding_eval) - len(completed))

Completed: 20
Remaining: 0


- the previous output only showed 6, but i ran it twice, all 20 queries saved

In [79]:
#check for missing predction
embedding_eval["llm_prediction"].isna().sum()

np.int64(0)

In [80]:
embedding_eval[
    [
        "technique_id",
        "embedding_retrieval_success",
        "llm_prediction",
        "llm_correct",
        "valid_candidate"
    ]
]

,technique_id,embedding_retrieval_success,llm_prediction,llm_correct,valid_candidate
2690,T1105,False,T1003,False,True
2650,T1003.001,False,T1136.002,False,True
2999,T1566.001,True,T1566.001,True,True
2821,T1059.003,False,T1134.002,False,True
2634,T1105,True,T1105,True,True
2739,T1003.001,True,T1003.001,True,True
2850,T1140,True,T1140,True,True
2966,T1059.003,True,T1059.003,True,True
2719,T1070.004,True,T1485,False,True
2721,T1083,False,T1561.001,False,True


In [81]:
#now overall accuracy
overall_accuracy = embedding_eval["llm_correct"].mean()

correct_count = embedding_eval["llm_correct"].sum()
total_count = len(embedding_eval)

print(f"Correct: {correct_count}/{total_count}")
print(f"Overall Embedding + LLM accuracy: {overall_accuracy:.3f}")

Correct: 7/20
Overall Embedding + LLM accuracy: 0.350


Embedding + Gemini 2.5 Flash: 7/20 correct = 35.0% accuracy
- the entire embedding retrieval + Gemini pipeline correctly mapped 35% of the 20 evaluation queries.

In [82]:
retrieval_successes = embedding_eval[
    embedding_eval["embedding_retrieval_success"]
].copy()

In [83]:
conditional_accuracy = (
    retrieval_successes["llm_correct"].mean()
)

conditional_correct = (
    retrieval_successes["llm_correct"].sum()
)

print(
    f"Correct when retrieval succeeded: "
    f"{conditional_correct}/{len(retrieval_successes)}"
)

print(
    f"Conditional LLM accuracy: "
    f"{conditional_accuracy:.3f}"
)

Correct when retrieval succeeded: 7/10
Conditional LLM accuracy: 0.700


- so Embedding retrieval success (Correct answer available) --> 10 queries --> Gemini chose correct answer --> 7 queries = 70%

### Embedding + LLM Results

The sentence-embedding retrieval pipeline combined with Gemini 2.5 Flash achieved an overall accuracy of **35% (7/20)**. However, the correct ATT&CK technique was present in the retrieved Top-5 candidates for only 10 of the 20 queries.

Among these 10 successful retrieval cases, the LLM selected the correct technique in **7 cases**, resulting in a **conditional LLM accuracy of 70%**. This suggests that the LLM was relatively effective at selecting the correct technique when retrieval provided it as a candidate, while retrieval failures remained a major constraint on overall pipeline performance.

## Current results

| Retrieval | Top-5 availability in sample | Overall LLM accuracy | Conditional LLM accuracy |
|---|---:|---:|---:|
| Embeddings | 10/20 | **35%** | **70%** |
| TF-IDF | ? | ? | ? |
| BoW | ? | ? | ? |

In [84]:
#generate tf-idf top 5 candidate with same 20 queries
tfidf_eval = evaluation_sample.copy()

In [85]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

#create vectorizer
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

In [86]:
#fit on retreival corpus
tfidf_mitre_matrix = tfidf_vectorizer.fit_transform(
    mitre_df["retrieval_text"]
)

In [87]:
#same 20 qurries
tfidf_query_matrix = tfidf_vectorizer.transform(
    tfidf_eval["sentence"]
)

In [88]:
#siilarity
tfidf_similarity = cosine_similarity(
    tfidf_query_matrix,
    tfidf_mitre_matrix
)

In [90]:
#top 5 tf-idf candidates
tfidf_candidates = []

for i in range(len(tfidf_eval)):

    scores = tfidf_similarity[i]

    top_indices = np.argsort(scores)[::-1][:5]

    candidates = []

    for idx in top_indices:

        candidates.append({
            "technique_id": mitre_df.iloc[idx]["technique_id"],
            "name": mitre_df.iloc[idx]["name"],
            "description": mitre_df.iloc[idx]["retrieval_text"],
            "score": scores[idx]
        })

    tfidf_candidates.append(candidates)

In [91]:
tfidf_eval["tfidf_candidates"] = tfidf_candidates

example = tfidf_eval.iloc[0]

print("Threat:")
print(example["sentence"])

print("\nGround truth:")
print(example["technique_id"])

print("\nTF-IDF Top 5:")

for candidate in example["tfidf_candidates"]:
    print(
        candidate["technique_id"],
        "-",
        candidate["name"]
    )

Threat:
Credential access permits the ransomware to deploy additional tools that further propagate the attack.

Ground truth:
T1105

TF-IDF Top 5:
T1610 - Deploy Container
T1679 - Selective Exclusion
T1486 - Data Encrypted for Impact
T1555.004 - Windows Credential Manager
T1003 - OS Credential Dumping


In [92]:
tfidf_eval["tfidf_retrieval_success"] = tfidf_eval.apply(
    lambda row: contains_ground_truth(
        row["tfidf_candidates"],
        row["technique_id"]
    ),
    axis=1
)

In [93]:
print(
    tfidf_eval["tfidf_retrieval_success"].value_counts()
)

tfidf_retrieval_success
False    16
True      4
Name: count, dtype: int64


**for the same 20 queries:**

- True  = 4 correct technique available to Gemini
- False = 16 correct technique absent from Top 5

In [94]:
#sample recall
tfidf_sample_recall5 = (
    tfidf_eval["tfidf_retrieval_success"].mean()
)

print(
    f"TF-IDF Recall@5 on LLM sample: "
    f"{tfidf_sample_recall5:.3f}"
)

TF-IDF Recall@5 on LLM sample: 0.200


- On the same 20-query sample, TF-IDF retrieved the correct ATT&CK technique in the Top 5 for only 4/20 queries (20%), compared with 10/20 (50%) for embeddings.

In [95]:
#prepare the tfidf result columns
tfidf_eval["llm_raw_response"] = None
tfidf_eval["llm_prediction"] = None
tfidf_eval["llm_correct"] = False
tfidf_eval["valid_candidate"] = False

In [96]:
tfidf_eval.to_pickle(
    "tfidf_llm_results.pkl"
)

print("TF-IDF evaluation setup saved.")

TF-IDF evaluation setup saved.


In [108]:
#same loop to gemini
import time

for i, row in tfidf_eval.iterrows():

    # Skip queries already completed
    if pd.notna(row["llm_prediction"]):
        continue

    prompt = build_prompt(
        row["sentence"],
        row["tfidf_candidates"]
    )

    try:
        raw_response = call_llm(prompt)

        prediction = extract_attack_id(
            raw_response
        )

        valid = prediction_in_candidates(
            prediction,
            row["tfidf_candidates"]
        )

        correct = (
            prediction == row["technique_id"]
        )

        # Store results
        tfidf_eval.at[i, "llm_raw_response"] = raw_response
        tfidf_eval.at[i, "llm_prediction"] = prediction
        tfidf_eval.at[i, "valid_candidate"] = valid
        tfidf_eval.at[i, "llm_correct"] = correct

        # Save after every API call
        tfidf_eval.to_pickle(
            "tfidf_llm_results.pkl"
        )

        print(
            f"Completed {i}: "
            f"prediction={prediction}, "
            f"correct={correct}"
        )

        time.sleep(15)

    except Exception as e:

        print(f"Stopped at row {i}")
        print("Error:", e)

        tfidf_eval.to_pickle(
            "tfidf_llm_results.pkl"
        )

        break

Completed 2742: prediction=T1020, correct=False


- the output again does not show all 20 queiries since i ran it twice. gemini quota runs out and i have to wait to run again

In [109]:
completed = tfidf_eval[
    tfidf_eval["llm_prediction"].notna()
]

remaining = tfidf_eval[
    tfidf_eval["llm_prediction"].isna()
]

print("Completed:", len(completed))
print("Remaining:", len(remaining))

Completed: 20
Remaining: 0


In [110]:
#accuracy
tfidf_overall_accuracy = (
    tfidf_eval["llm_correct"].mean()
)

tfidf_correct_count = (
    tfidf_eval["llm_correct"].sum()
)

print(
    f"Correct: {tfidf_correct_count}/20"
)

print(
    f"Overall TF-IDF + LLM accuracy: "
    f"{tfidf_overall_accuracy:.3f}"
)

Correct: 5/20
Overall TF-IDF + LLM accuracy: 0.250


In [111]:
tfidf_retrieval_successes = tfidf_eval[
    tfidf_eval["tfidf_retrieval_success"]
].copy()

In [112]:
tfidf_conditional_accuracy = (
    tfidf_retrieval_successes["llm_correct"].mean()
)

tfidf_conditional_correct = (
    tfidf_retrieval_successes["llm_correct"].sum()
)

print(
    f"Correct when retrieval succeeded: "
    f"{tfidf_conditional_correct}/"
    f"{len(tfidf_retrieval_successes)}"
)

print(
    f"Conditional LLM accuracy: "
    f"{tfidf_conditional_accuracy:.3f}"
)

Correct when retrieval succeeded: 4/4
Conditional LLM accuracy: 1.000


**so far**

| Retrieval method | Correct in Top 5 | Overall LLM accuracy | Conditional LLM accuracy |
|---|---:|---:|---:|
| Embeddings | 10/20 | 7/20 = **35%** | 7/10 = **70%** |
| TF-IDF | 4/20 | 4/20 = **20%** | 4/4 = **100%** |

Embedding retrieval gave the LLM substantially more opportunities to make the correct classification (10 versus 4). Gemini selected all four correct candidates supplied by TF-IDF, while it selected 7 of the 10 supplied by embeddings. As a result, the embedding-based pipeline still achieved higher overall accuracy: 35% versus 20%.

In [113]:
#save tf-idf result
tfidf_eval.to_pickle(
    "tfidf_llm_results.pkl"
)

tfidf_eval.to_csv(
    "tfidf_llm_results.csv",
    index=False
)

print("TF-IDF LLM results saved.")

TF-IDF LLM results saved.


**Moving on to Bow**

same process

In [35]:
bow_eval = evaluation_sample.copy()

In [36]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

bow_vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english"
)

bow_mitre_matrix = bow_vectorizer.fit_transform(
    mitre_df["retrieval_text"]
)

bow_query_matrix = bow_vectorizer.transform(
    bow_eval["sentence"]
)

bow_similarity = cosine_similarity(
    bow_query_matrix,
    bow_mitre_matrix
)

In [37]:
#extract top 5 candidates
bow_candidates = []

for i in range(len(bow_eval)):

    scores = bow_similarity[i]

    top_indices = np.argsort(scores)[::-1][:5]

    candidates = []

    for idx in top_indices:

        candidates.append({
            "technique_id": mitre_df.iloc[idx]["technique_id"],
            "name": mitre_df.iloc[idx]["name"],
            "description": mitre_df.iloc[idx]["retrieval_text"],
            "score": scores[idx]
        })

    bow_candidates.append(candidates)

In [38]:
bow_eval["bow_candidates"] = bow_candidates

In [39]:
example = bow_eval.iloc[0]

print("Threat:")
print(example["sentence"])

print("\nGround truth:")
print(example["technique_id"])

print("\nBoW Top 5:")

for candidate in example["bow_candidates"]:
    print(
        candidate["technique_id"],
        "-",
        candidate["name"]
    )

Threat:
Credential access permits the ransomware to deploy additional tools that further propagate the attack.

Ground truth:
T1105

BoW Top 5:
T1003 - OS Credential Dumping
T1556.009 - Conditional Access Policies
T1650 - Acquire Access
T1219 - Remote Access Tools
T1136 - Create Account


In [40]:
bow_eval["bow_retrieval_success"] = bow_eval.apply(
    lambda row: contains_ground_truth(
        row["bow_candidates"],
        row["technique_id"]
    ),
    axis=1
)

print(
    bow_eval["bow_retrieval_success"].value_counts()
)

bow_retrieval_success
False    15
True      5
Name: count, dtype: int64


In [41]:
#calculate recall for same 20 queries
bow_sample_recall5 = (
    bow_eval["bow_retrieval_success"].mean()
)

print(
    f"BoW Recall@5 on LLM sample: "
    f"{bow_sample_recall5:.3f}"
)

BoW Recall@5 on LLM sample: 0.250


## summary 

| Method | Top-5 retrieval success |
|---|---:|
| Embeddings | **10/20 = 50%** |
| BoW | **5/20 = 25%** |
| TF-IDF | **4/20 = 20%** |

In [42]:
#Bow to Gemini
bow_eval["llm_raw_response"] = None
bow_eval["llm_prediction"] = None
bow_eval["llm_correct"] = False
bow_eval["valid_candidate"] = False

In [131]:
bow_eval.to_pickle(
    "bow_llm_results.pkl"
)

print("BoW evaluation setup saved.")

BoW evaluation setup saved.


In [50]:
import time

for i, row in bow_eval.iterrows():

    # Skip already completed queries
    if pd.notna(row["llm_prediction"]):
        continue

    prompt = build_prompt(
        row["sentence"],
        row["bow_candidates"]
    )

    try:
        raw_response = call_llm(prompt)

        prediction = extract_attack_id(
            raw_response
        )

        valid = prediction_in_candidates(
            prediction,
            row["bow_candidates"]
        )

        # Count as correct ONLY if Gemini selected
        # a supplied candidate AND it matches ground truth
        correct = (
            prediction == row["technique_id"]
            and valid
        )

        bow_eval.at[i, "llm_raw_response"] = raw_response
        bow_eval.at[i, "llm_prediction"] = prediction
        bow_eval.at[i, "valid_candidate"] = valid
        bow_eval.at[i, "llm_correct"] = correct

        # Save after every successful request
        bow_eval.to_pickle(
            "bow_llm_results.pkl"
        )

        print(
            f"Completed {i}: "
            f"prediction={prediction}, "
            f"correct={correct}"
        )

        time.sleep(15)

    except Exception as e:

        print(f"Stopped at row {i}")
        print("Error:", e)

        bow_eval.to_pickle(
            "bow_llm_results.pkl"
        )

        break

Completed 2965: prediction=T1588.006, correct=False
Completed 2644: prediction=T1611, correct=False
Completed 2908: prediction=T1059.003, correct=True
Completed 2621: prediction=T1574.001, correct=False
Completed 2639: prediction=T1190, correct=False
Completed 2742: prediction=T1020, correct=False


In [48]:
import pandas as pd
#load save results
bow_eval = pd.read_pickle(
    "bow_llm_results.pkl"
)

- same process as before.
- load saved results then rerun gemini due to quota. the output wont show all results

In [51]:
completed = bow_eval[
    bow_eval["llm_prediction"].notna()
]

remaining = bow_eval[
    bow_eval["llm_prediction"].isna()
]

print("Completed:", len(completed))
print("Remaining:", len(remaining))

Completed: 20
Remaining: 0


Once all 20 finish, tell me. Then we'll calculate BoW overall accuracy and conditional LLM accuracy, and finally create the complete comparison of all three RAG pipelines.

In [53]:
invalid_bow = bow_eval[
    ~bow_eval["valid_candidate"]
].copy()

invalid_bow[
    [
        "technique_id",
        "bow_retrieval_success",
        "llm_prediction",
        "llm_correct",
        "llm_raw_response"
    ]
]

,technique_id,bow_retrieval_success,llm_prediction,llm_correct,llm_raw_response
2639,T1190,False,T1190,False,"The cyber threat description states: ""Threat A..."


In [54]:
for idx, row in invalid_bow.iterrows():

    print("Row:", idx)
    print("Ground truth:", row["technique_id"])
    print("Gemini prediction:", row["llm_prediction"])
    print(
        "Retrieval success:",
        row["bow_retrieval_success"]
    )

    print("\nAllowed candidates:")

    for candidate in row["bow_candidates"]:
        print(
            candidate["technique_id"],
            "-",
            candidate["name"]
        )

Row: 2639
Ground truth: T1190
Gemini prediction: T1190
Retrieval success: False

Allowed candidates:
T1668 - Exclusive Control
T1681 - Search Threat Vendor Data
T1677 - Poisoned Pipeline Execution
T1597.001 - Threat Intel Vendors
T1204.005 - Malicious Library


In [55]:
bow_eval["llm_correct"] = (
    (bow_eval["llm_prediction"] == bow_eval["technique_id"])
    &
    (bow_eval["valid_candidate"])
)

In [56]:
#accuracy
bow_correct_count = bow_eval["llm_correct"].sum()
bow_overall_accuracy = bow_eval["llm_correct"].mean()

print(f"Correct: {bow_correct_count}/20")
print(
    f"Overall BoW + LLM accuracy: "
    f"{bow_overall_accuracy:.3f}"
)

Correct: 4/20
Overall BoW + LLM accuracy: 0.200


In [57]:
bow_successes = bow_eval[
    bow_eval["bow_retrieval_success"]
]

bow_conditional_correct = (
    bow_successes["llm_correct"].sum()
)

bow_conditional_accuracy = (
    bow_successes["llm_correct"].mean()
)

print(
    f"Correct when retrieval succeeded: "
    f"{bow_conditional_correct}/"
    f"{len(bow_successes)}"
)

print(
    f"Conditional LLM accuracy: "
    f"{bow_conditional_accuracy:.3f}"
)

Correct when retrieval succeeded: 4/5
Conditional LLM accuracy: 0.800


## final comparison

| Pipeline | Correct in Top 5 | Retrieval Recall@5 | Overall LLM Accuracy | Conditional LLM Accuracy |
|---|---:|---:|---:|---:|
| Bag-of-Words + Gemini | 5/20 | 25% | **20% (4/20)** | **80% (4/5)** |
| TF-IDF + Gemini | 4/20 | 20% | **20% (4/20)** | **100% (4/4)** |
| Sentence Embeddings + Gemini | 10/20 | **50%** | **35% (7/20)** | **70% (7/10)** |

### LLM-Assisted ATT&CK Mapping Results

To evaluate the effect of retrieval quality on downstream ATT&CK mapping, Gemini 2.5 Flash was provided with the Top-5 candidates produced by each retrieval method for the same 20 threat descriptions. The LLM was instructed to select one technique exclusively from the retrieved candidates.

The sentence-embedding pipeline achieved the highest overall accuracy at **35% (7/20)**, compared with **20% (4/20)** for both Bag-of-Words and TF-IDF. Embedding retrieval placed the correct technique within the Top 5 for 10 of the 20 queries, compared with 5 for Bag-of-Words and 4 for TF-IDF.

When the correct technique was available among the retrieved candidates, Gemini selected it correctly in **7/10 embedding cases, 4/5 Bag-of-Words cases, and 4/4 TF-IDF cases**. However, these conditional results are based on small numbers of successful retrievals and should be interpreted cautiously.

Overall, the results suggest that **retrieval quality is an important constraint on downstream LLM-based ATT&CK mapping**. Sentence embeddings provided the LLM with the correct candidate more frequently and consequently produced the strongest end-to-end performance, supporting the hypothesis that improved retrieval can lead to improved LLM classification accuracy.

In [59]:
# Save final per-query LLM results

bow_eval.to_pickle("bow_llm_results.pkl")
bow_eval.to_csv("bow_llm_results.csv", index=False)

print("All final LLM query results saved.")

All final LLM query results saved.


In [61]:
import pandas as pd

bow_eval = pd.read_pickle(
    "bow_llm_results.pkl"
)

tfidf_eval = pd.read_pickle(
    "tfidf_llm_results.pkl"
)

embedding_eval = pd.read_pickle(
    "embedding_llm_results.pkl"
)

In [62]:
#save the final comparison table
comparison = pd.DataFrame({
    "Pipeline": [
        "Bag-of-Words + Gemini",
        "TF-IDF + Gemini",
        "Sentence Embeddings + Gemini"
    ],

    "Retrieval Recall@5": [
        bow_eval["bow_retrieval_success"].mean(),
        tfidf_eval["tfidf_retrieval_success"].mean(),
        embedding_eval["embedding_retrieval_success"].mean()
    ],

    "Overall LLM Accuracy": [
        bow_eval["llm_correct"].mean(),
        tfidf_eval["llm_correct"].mean(),
        embedding_eval["llm_correct"].mean()
    ],

    "Conditional LLM Accuracy": [
        bow_eval.loc[
            bow_eval["bow_retrieval_success"],
            "llm_correct"
        ].mean(),

        tfidf_eval.loc[
            tfidf_eval["tfidf_retrieval_success"],
            "llm_correct"
        ].mean(),

        embedding_eval.loc[
            embedding_eval["embedding_retrieval_success"],
            "llm_correct"
        ].mean()
    ]
})

comparison.to_csv(
    "llm_pipeline_comparison.csv",
    index=False
)

comparison

,Pipeline,Retrieval Recall@5,Overall LLM Accuracy,Conditional LLM Accuracy
0,Bag-of-Words + Gemini,0.25,0.20,0.8
1,TF-IDF + Gemini,0.20,0.25,1.0
2,Sentence Embeddings + Gemini,0.50,0.35,0.7
